In [3]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.base.model import Model
from scipy import stats

In [4]:
df = pd.read_csv('bank_case_study-Table 1.csv')
df

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4516,33,services,married,secondary,no,-333,yes,no,cellular,30,jul,329,5,-1,0,unknown,no
4517,57,self-employed,married,tertiary,yes,-3313,yes,yes,unknown,9,may,153,1,-1,0,unknown,no
4518,57,technician,married,secondary,no,295,no,no,cellular,19,aug,151,11,-1,0,unknown,no
4519,28,blue-collar,married,secondary,no,1137,no,no,cellular,6,feb,129,4,211,3,other,no


In [5]:
df.isnull().sum()

,0
age,0
job,0
marital,0
education,0
default,0
balance,0
housing,0
loan,0
contact,0
day,0


In [6]:
fit5=ols('balance ~ job+marital+education+default+housing+loan+contact+day+month+duration+campaign+pdays+previous+poutcome+y',data=df).fit()
annova5=sm.stats.anova_lm(fit5,typ=1)
annova5

,df,sum_sq,mean_sq,F,PR(>F)
job,11.0,5.477477e+08,4.979524e+07,5.767308,2.493381e-09
marital,2.0,6.649770e+07,3.324885e+07,3.850898,2.133107e-02
education,3.0,1.001948e+08,3.339826e+07,3.868202,8.924435e-03
default,1.0,1.864908e+08,1.864908e+08,21.599451,3.455986e-06
housing,1.0,1.874895e+07,1.874895e+07,2.171512,1.406582e-01
loan,1.0,1.465342e+08,1.465342e+08,16.971666,3.862673e-05
contact,2.0,2.376117e+07,1.188058e+07,1.376015,2.526899e-01
month,11.0,1.151518e+09,1.046835e+08,12.124491,8.436481e-23
poutcome,3.0,7.235984e+06,2.411995e+06,0.279358,8.403363e-01
y,1.0,8.493004e+06,8.493004e+06,0.983664,3.213495e-01


In [10]:
fit4=ols('balance ~ education*loan',data=df).fit()
annova4=sm.stats.anova_lm(fit4,typ=1)
annova4

,df,sum_sq,mean_sq,F,PR(>F)
education,3.0,3.002143e+08,1.000714e+08,11.184054,2.611309e-07
loan,1.0,1.738790e+08,1.738790e+08,19.432847,1.066205e-05
education:loan,3.0,8.679990e+07,2.893330e+07,3.233606,2.137920e-02
Residual,4513.0,4.038091e+10,8.947687e+06,NaN,NaN


In [11]:
tukey=pairwise_tukeyhsd(df["balance"],groups=df["education"]+df["loan"].astype(str))
tukey._results_table

group1,group2,meandiff,p-adj,lower,upper,reject
primaryno,primaryyes,-1005.6023,0.0511,-2013.6319,2.4272,False
primaryno,secondaryno,-263.1672,0.5799,-692.5961,166.2617,False
primaryno,secondaryyes,-767.5073,0.0017,-1349.4439,-185.5707,True
primaryno,tertiaryno,294.3183,0.5207,-164.8533,753.4899,False
primaryno,tertiaryyes,-247.6882,0.98,-1031.0859,535.7096,False
primaryno,unknownno,50.8415,1.0,-722.4313,824.1144,False
primaryno,unknownyes,2707.3217,0.2512,-741.4708,6156.1141,False
primaryyes,secondaryno,742.4351,0.2674,-216.0928,1700.963,False
primaryyes,secondaryyes,238.095,0.9971,-797.7702,1273.9603,False
primaryyes,tertiaryno,1299.9206,0.0013,327.704,2272.1372,True


In [12]:
tukey_df = pd.DataFrame(
    tukey._results_table.data[1:],
    columns=tukey._results_table.data[0]
)
print(tukey_df[tukey_df["reject"] == True])

          group1        group2   meandiff   p-adj      lower      upper  \
2      primaryno  secondaryyes  -767.5073  0.0017 -1349.4439  -185.5707   
9     primaryyes    tertiaryno  1299.9206  0.0013   327.7040  2272.1372   
12    primaryyes    unknownyes  3712.9240  0.0332   159.2589  7266.5891   
13   secondaryno  secondaryyes  -504.3401  0.0393  -995.5655   -13.1147   
14   secondaryno    tertiaryno   557.4855  0.0000   220.6016   894.3694   
18  secondaryyes    tertiaryno  1061.8256  0.0000   544.3975  1579.2536   
20  secondaryyes     unknownno   818.3488  0.0452     9.1244  1627.5732   
21  secondaryyes    unknownyes  3474.8290  0.0478    17.7981  6931.8599   

    reject  
2     True  
9     True  
12    True  
13    True  
14    True  
18    True  
20    True  
21    True  


In [22]:
g1 = df[(df["education"] == "primary") & (df["loan"] == "yes")]["balance"]
g2 = df[(df["education"] == "unknown") & (df["loan"] == "yes")]["balance"]

stats.ttest_ind(g1, g2, equal_var=True,alternative="greater")

TtestResult(statistic=np.float64(-4.273243695918309), pvalue=np.float64(0.999977823734127), df=np.float64(99.0))

In [ ]:
#Customers with primary education who subscribed (yes) do not have significantly
#higher balance than customers with unknown education who subscribed (yes).

In [23]:
df[(df["education"] == "unknown") & (df["loan"] == "yes")]

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
304,56,management,married,unknown,no,353,no,yes,cellular,25,jul,171,2,-1,0,unknown,no
1069,45,technician,single,unknown,no,7108,yes,yes,cellular,18,nov,53,1,172,3,failure,no
1561,54,services,married,unknown,no,386,no,yes,cellular,19,nov,74,1,152,1,success,no
1821,51,housemaid,married,unknown,no,21244,no,yes,cellular,4,aug,166,2,-1,0,unknown,no
3194,28,management,single,unknown,no,81,no,yes,cellular,8,jul,158,1,-1,0,unknown,no
4279,31,housemaid,married,unknown,yes,-6,no,yes,telephone,7,jul,94,2,-1,0,unknown,no
4508,42,admin.,married,unknown,no,642,yes,yes,unknown,16,may,509,2,-1,0,unknown,no


In [24]:
df[(df["education"] == "unknown") & (df["loan"] == "yes")].shape

(7, 17)

In [25]:
df[(df["education"] == "primary") & (df["loan"] == "yes")].shape

(94, 17)

In [ ]:
#Fewer customers have unknown education with loan = yes (7) compared to primary
#education with loan = yes (94). This imbalance occurs because more customers in
#the dataset reported their education level as primary, while only a small number
#fall under the unknown education category.